In [ ]:
%cd ../../demo

# User Guide
## Installation
Download the package very simply with pip

```
pip install cpys
```

## Preliminaries
1. You need a csv file which contains your track(s) (`tracks.csv` hereafter, but you can name it as you wish). This csv can be the output of TempestExtremes' StitchNodes, or it can be obtained from other trackers (e.g. TRACK). It needs to contains at least the longitude and the latitude of the points, named `lon` and `lat`. If several tracks are in the file, it is advised to have a `track_id` variable to distinguish among the tracks. Otherwise, the value of $\theta$ and $B$ for the last point of each track will be wrong. If your csv file has any other additional columns, they will be kept through the process, although not used.
2. You need a NetCDF file with the geopotential at several levels from 900 to 300hPa (at least five are recommended) covering the region where your tracks are (`geopt.nc` hereafter, but you can name it as you wish).
3. Use TempestExtremes' NodeFileCompose to obtain snapshots of the geopotential field along the track(s). Use a code similar to that below, adapted to your data:

```
NodeFileCompose \
    --in_nodefile "tracks.csv" \
    --in_nodefile_type SN \
    --in_fmt "(auto)" \
    --in_data "geopt.nc" \
    # Use radial grid for circular snapshots along the tracks
    --out_grid "RAD" \
    # Snapshots dimension along the radial axis : 10 steps every 0.5° = 500km
    --dx 0.5 --resx 10  \
    --out_data "snaps.nc" \
    # Change z to the name of the geopt variable in your data
    --var "z(:)" \
    # Your snapshots will be named "snap_zg"
    --varout "zg" \
    # Make sure to output individual snapshots
    --snapshots \
    # Change to the names in your geopt.nc file
    --latname latitude --lonname longitude \
    # Use this option if your geopt.nc file is not global
    --regional
```

**NB : NodeFileCompose does not output the value of the vertical coordinate. Be careful to change it before using the snapshots with CPyS**

*This example is based on the track of Typhoon Dale. `Dale.csv` contains the track data, and `Dale.nc` contains the snapshots.*

## Loading the data

In [ ]:
# Load the csv data using huracanpy
import huracanpy
track = huracanpy.load("Dale.csv")
track[["track_id", "time", "lon", "lat"]]

In [ ]:
# Load the snapshots file with xarray
import xarray as xr
snaps = xr.open_dataset("Dale.nc").snap_zg
# !! Change the vertical variable here if necessary. It must be in Pa
#snaps["level"] = [...]

In [ ]:
# Snapshots dimensions
snaps.dims

In [ ]:
# Dimensions
snaps.coords

In [ ]:
# Computation of the CPS parameters
from cpys import compute_cps_parameters

track_w_cps_params = compute_cps_parameters(track, snaps)

# Results!
track_w_cps_params[["track_id", "time", "lon", "lat", "theta", "B", "VTL", "VTU"]]

## Plot of the phase space diagram
I have included a simple function to plot the two traditionnal phase space diagrams.

In [ ]:
from cpys import plot_cps

plot_cps(track_w_cps_params, title = "Dale")